# Feature Engineering

In [2406]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import PowerTransformer
from category_encoders import TargetEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant


In [2407]:
def zero_pct(df, cols):

    df_zero = (df[cols] == 0).mean().to_frame('ZERO_PCT').sort_values('ZERO_PCT', ascending=False)
    df_zero = df_zero[df_zero['ZERO_PCT'] > 0]
    return df_zero

In [2408]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / 'data'
CLEAN_DATA_DIR = DATA_DIR / 'data_clean'

In [2409]:
# TODO: revisit this read approach — see cleaning.py for a cleaner pattern
df_clean = pd.read_csv(CLEAN_DATA_DIR / 'train_clean.csv', keep_default_na=False, na_values=[''])

# Classification and Overview

In [2410]:
df_clean.head()

,ID,MSSUBCLASS,MSZONING,LOTFRONTAGE,LOTAREA,STREET,ALLEY,LOTSHAPE,LANDCONTOUR,UTILITIES,LOTCONFIG,LANDSLOPE,NEIGHBORHOOD,CONDITION1,CONDITION2,BLDGTYPE,HOUSESTYLE,OVERALLQUAL,OVERALLCOND,YEARBUILT,YEARREMODADD,ROOFSTYLE,ROOFMATL,EXTERIOR1ST,EXTERIOR2ND,MASVNRTYPE,MASVNRAREA,EXTERQUAL,EXTERCOND,FOUNDATION,BSMTQUAL,BSMTCOND,BSMTEXPOSURE,BSMTFINTYPE1,BSMTFINSF1,BSMTFINTYPE2,BSMTFINSF2,BSMTUNFSF,TOTALBSMTSF,HEATING,HEATINGQC,CENTRALAIR,ELECTRICAL,1STFLRSF,2NDFLRSF,LOWQUALFINSF,GRLIVAREA,BSMTFULLBATH,BSMTHALFBATH,FULLBATH,HALFBATH,BEDROOMABVGR,KITCHENABVGR,KITCHENQUAL,TOTRMSABVGRD,FUNCTIONAL,FIREPLACES,FIREPLACEQU,GARAGETYPE,GARAGEFINISH,GARAGECARS,GARAGEAREA,GARAGEQUAL,GARAGECOND,PAVEDDRIVE,WOODDECKSF,OPENPORCHSF,ENCLOSEDPORCH,3SSNPORCH,SCREENPORCH,FENCE,MISCFEATURE,MISCVAL,MOSOLD,YRSOLD,SALETYPE,SALECONDITION,SALEPRICE,HASGARAGE,HASPOOL
0,1,60,RL,65.0,8450.0,PAVE,NA,REG,LVL,ALLPUB,INSIDE,GTL,COLLGCR,NORM,NORM,1FAM,2STORY,7,5,2003,2003,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,196.0,GD,TA,PCONC,GD,TA,NO,GLQ,706.0,UNF,0.0,150.0,856.0,GASA,EX,Y,SBRKR,856.0,854.0,0.0,1710.0,1,0,2,1,3,1,GD,8,TYP,0,NA,ATTCHD,RFN,2,548.0,TA,TA,Y,0.0,61.0,0.0,0.0,0.0,NA,NA,0,2,2008,WD,NORMAL,208500.0,True,False
1,2,20,RL,80.0,9600.0,PAVE,NA,REG,LVL,ALLPUB,FR2,GTL,VEENKER,FEEDR,NORM,1FAM,1STORY,6,8,1976,1976,GABLE,COMPSHG,METALSD,METALSD,None,0.0,TA,TA,CBLOCK,GD,TA,GD,ALQ,978.0,UNF,0.0,284.0,1262.0,GASA,EX,Y,SBRKR,1262.0,0.0,0.0,1262.0,0,1,2,0,3,1,TA,6,TYP,1,TA,ATTCHD,RFN,2,460.0,TA,TA,Y,298.0,0.0,0.0,0.0,0.0,NA,NA,0,5,2007,WD,NORMAL,181500.0,True,False
2,3,60,RL,68.0,11250.0,PAVE,NA,IR1,LVL,ALLPUB,INSIDE,GTL,COLLGCR,NORM,NORM,1FAM,2STORY,7,5,2001,2002,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,162.0,GD,TA,PCONC,GD,TA,MN,GLQ,486.0,UNF,0.0,434.0,920.0,GASA,EX,Y,SBRKR,920.0,866.0,0.0,1786.0,1,0,2,1,3,1,GD,6,TYP,1,TA,ATTCHD,RFN,2,608.0,TA,TA,Y,0.0,42.0,0.0,0.0,0.0,NA,NA,0,9,2008,WD,NORMAL,223500.0,True,False
3,4,70,RL,60.0,9550.0,PAVE,NA,IR1,LVL,ALLPUB,CORNER,GTL,CRAWFOR,NORM,NORM,1FAM,2STORY,7,5,1915,1970,GABLE,COMPSHG,WD SDNG,WD SHNG,None,0.0,TA,TA,BRKTIL,TA,GD,NO,ALQ,216.0,UNF,0.0,540.0,756.0,GASA,GD,Y,SBRKR,961.0,756.0,0.0,1717.0,1,0,1,0,3,1,GD,7,TYP,1,GD,DETCHD,UNF,3,642.0,TA,TA,Y,0.0,35.0,272.0,0.0,0.0,NA,NA,0,2,2006,WD,ABNORML,140000.0,True,False
4,5,60,RL,84.0,14260.0,PAVE,NA,IR1,LVL,ALLPUB,FR2,GTL,NORIDGE,NORM,NORM,1FAM,2STORY,8,5,2000,2000,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,350.0,GD,TA,PCONC,GD,TA,AV,GLQ,655.0,UNF,0.0,490.0,1145.0,GASA,EX,Y,SBRKR,1145.0,1053.0,0.0,2198.0,1,0,2,1,4,1,GD,9,TYP,1,TA,ATTCHD,RFN,3,836.0,TA,TA,Y,192.0,84.0,0.0,0.0,0.0,NA,NA,0,12,2008,WD,NORMAL,250000.0,True,False


In [2411]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1412 entries, 0 to 1411
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ID             1412 non-null   int64  
 1   MSSUBCLASS     1412 non-null   int64  
 2   MSZONING       1412 non-null   str    
 3   LOTFRONTAGE    1412 non-null   float64
 4   LOTAREA        1412 non-null   float64
 5   STREET         1412 non-null   str    
 6   ALLEY          1412 non-null   str    
 7   LOTSHAPE       1412 non-null   str    
 8   LANDCONTOUR    1412 non-null   str    
 9   UTILITIES      1412 non-null   str    
 10  LOTCONFIG      1412 non-null   str    
 11  LANDSLOPE      1412 non-null   str    
 12  NEIGHBORHOOD   1412 non-null   str    
 13  CONDITION1     1412 non-null   str    
 14  CONDITION2     1412 non-null   str    
 15  BLDGTYPE       1412 non-null   str    
 16  HOUSESTYLE     1412 non-null   str    
 17  OVERALLQUAL    1412 non-null   int64  
 18  OVERALLCOND    1412

In [2412]:
df_clean.columns = df_clean.columns.str.upper()

df_clean['TOTALBATHS'] = df_clean['FULLBATH'] + (0.5*df_clean['HALFBATH']) + df_clean['BSMTFULLBATH'] + (0.5*df_clean['BSMTHALFBATH'])

df_clean.drop(columns=['FULLBATH', 'HALFBATH', 'BSMTFULLBATH', 'BSMTHALFBATH'], inplace=True)

df_clean.drop(columns=['GARAGEAREA', 'GARAGECARS'], inplace=True)

df_clean['YEARBUILT'] = df_clean['YRSOLD'] - df_clean['YEARBUILT']
df_clean['YEARREMODADD'] = df_clean['YRSOLD'] - df_clean['YEARREMODADD']

df_clean.drop(columns=['YRSOLD', 'MOSOLD'], inplace=True)

df_clean['BSMTFINSF'] = df_clean['BSMTFINSF1'] + df_clean['BSMTFINSF2']
df_clean['TOTALFLRSF'] = df_clean['1STFLRSF'] + df_clean['2NDFLRSF']

df_clean.drop(columns=['BSMTFINSF1', 'BSMTFINSF2', '1STFLRSF', '2NDFLRSF'], inplace=True)

In [2413]:
ord_cols = [
    'LOTSHAPE', 'UTILITIES', 'LANDSLOPE', 'OVERALLQUAL',
    'OVERALLCOND', 'EXTERQUAL', 'EXTERCOND', 'BSMTQUAL',
    'BSMTCOND', 'BSMTEXPOSURE', 'BSMTFINTYPE1', 'BSMTFINTYPE2',
    'HEATINGQC', 'ELECTRICAL', 'KITCHENQUAL', 'FUNCTIONAL',
    'FIREPLACEQU', 'GARAGEFINISH', 'GARAGEQUAL', 'GARAGECOND',
    'PAVEDDRIVE', 'FENCE'
]

nom_cols = [
    'MSSUBCLASS', 'MSZONING', 'STREET', 'ALLEY', 'LANDCONTOUR',
    'LOTCONFIG', 'NEIGHBORHOOD', 'CONDITION1', 'CONDITION2',
    'BLDGTYPE', 'HOUSESTYLE', 'ROOFSTYLE', 'ROOFMATL',
    'EXTERIOR1ST', 'EXTERIOR2ND', 'MASVNRTYPE', 'FOUNDATION',
    'HEATING', 'CENTRALAIR', 'GARAGETYPE', 'MISCFEATURE',
    'SALETYPE', 'SALECONDITION'
]


disc_cols = df_clean.select_dtypes(include='int64').drop(
    columns=['ID'] + ord_cols + nom_cols, errors='ignore'
)


cont_cols = list(df_clean.select_dtypes(include='float64').drop(columns=['SALEPRICE'], errors='ignore').columns)
bool_cols = list(df_clean.select_dtypes(include='bool').columns)


# Qualitative ordinal

In [2414]:
df_ord = df_clean.copy()

for col in ord_cols:
    print(df_ord[col].value_counts().sort_index(ascending=False))
    print()

LOTSHAPE
REG    886
IR3     10
IR2     40
IR1    476
Name: count, dtype: int64

UTILITIES
NOSEWA       1
ALLPUB    1411
Name: count, dtype: int64

LANDSLOPE
SEV      13
MOD      64
GTL    1335
Name: count, dtype: int64

OVERALLQUAL
10     17
9      43
8     166
7     313
6     371
5     381
4     103
3      14
2       3
1       1
Name: count, dtype: int64

OVERALLCOND
9     22
8     72
7    200
6    248
5    792
4     52
3     20
2      5
1      1
Name: count, dtype: int64

EXTERQUAL
TA    871
GD    478
FA     12
EX     51
Name: count, dtype: int64

EXTERCOND
TA    1239
PO       1
GD     145
FA      24
EX       3
Name: count, dtype: int64

BSMTQUAL
TA    648
GD    609
FA     35
EX    120
Name: count, dtype: int64

BSMTCOND
TA    1301
PO       2
GD      64
FA      45
Name: count, dtype: int64

BSMTEXPOSURE
NO    944
MN    114
GD    133
AV    221
Name: count, dtype: int64

BSMTFINTYPE1
UNF    426
REC    132
LWQ     74
GLQ    412
BLQ    148
ALQ    220
Name: count, dtype: int64

BSMTFINTYP

In [2415]:
drop_cols = ['UTILITIES']
add_cols = []

# GRADES 1-10
grade_map = {
    0: 1,
    1: 1,
    2: 1,
    3: 1,
    4: 1,
    5: 2,
    6: 2,
    7: 2,
    8: 3,
    9: 3,
    10: 3,
}
df_ord['OVERALLCOND'] = df_ord['OVERALLCOND'].map(grade_map)
df_ord['OVERALLQUAL'] = df_ord['OVERALLQUAL'].map(grade_map)

# FIVE QUALITIES
qual5_map = {
    'NA': 0,
    'PO': 0,
    'FA': 2,
    'TA': 2,
    'GD': 3,
    'EX': 3,
}
df_ord['EXTERQUAL'] = df_ord['EXTERQUAL'].map(qual5_map)
df_ord['EXTERCOND'] = df_ord['EXTERCOND'].map(qual5_map)
df_ord['HEATINGQC'] = df_ord['HEATINGQC'].map(qual5_map)
df_ord['KITCHENQUAL'] = df_ord['KITCHENQUAL'].map(qual5_map)

# SIX QUALITIES
qual6_map = {'NA': 0, 'PO': 0, 'FA': 2, 'TA': 2, 'GD': 3, 'EX': 3}

df_ord['BSMTQUAL'] = df_ord['BSMTQUAL'].map(qual6_map)
df_ord['BSMTCOND'] = df_ord['BSMTCOND'].map(qual6_map)
df_ord['FIREPLACEQU'] = df_ord['FIREPLACEQU'].map(qual6_map)
df_ord['GARAGEQUAL'] = df_ord['GARAGEQUAL'].map(qual6_map)
df_ord['GARAGECOND'] = df_ord['GARAGECOND'].map(qual6_map)

# BSMT TYPE
bsmt_fin_map = {
    'NA': 0,
    'UNF': 1,
    'LWQ': 2,
    'REC': 2,
    'BLQ': 2,
    'ALQ': 3,
    'GLQ': 3,
}
df_ord['BSMTFINTYPE1'] = df_ord['BSMTFINTYPE1'].map(bsmt_fin_map)
df_ord['BSMTFINTYPE2'] = df_ord['BSMTFINTYPE2'].map(bsmt_fin_map)

# SPECIAL CASES
df_ord['LANDSLOPE'] = df_ord['LANDSLOPE'].map(
    {'NA': 0, 'SEV': 3, 'MOD': 2, 'GTL': 1}
)
df_ord['BSMTEXPOSURE'] = df_ord['BSMTEXPOSURE'].map(
    {'NA': 0, 'NO': 1, 'MN': 1, 'AV': 1, 'GD': 2}
)
df_ord['GARAGEFINISH'] = df_ord['GARAGEFINISH'].map(
    {'NA': 0, 'UNF': 1, 'RFN': 2, 'FIN': 3}
)

df_ord['ELECTRICAL'] = df_ord['ELECTRICAL'].map(
    {'MIX': 1, 'FUSEP': 1, 'FUSEF': 2, 'FUSEA': 2, 'SBRKR': 3}
)

df_ord['FENCE'] = df_ord['FENCE'].map(
    {'NA': 0, 'MNWW': 1, 'GDWO': 1, 'MNPRV': 1, 'GDPRV': 1}
)

df_ord['FUNCTIONAL'] = df_ord['FUNCTIONAL'].map(
    {
        'SEV': 0,
        'MAJ2': 0,
        'MAJ1': 0,
        'MOD': 1,
        'MIN2': 1,
        'MIN1': 1,
        'TYP': 2,
        'NA': 0,
    }
)

# BOOLEAN CONVERSION
df_ord = df_ord.rename(columns={'FENCE': 'HASFENCE'})
df_ord['HASFENCE'] = df_ord['HASFENCE'].astype('bool')
drop_cols.append('FENCE')
add_cols.append('HASFENCE')

df_ord['PAVEDDRIVE'] = df_ord['PAVEDDRIVE'].map(
    {'NA': 0, 'MNWW': 1, 'GDWO': 1, 'MNPRV': 1, 'GDPRV': 1}
)
df_ord = df_ord.rename(columns={'PAVEDDRIVE': 'HASPAVEDDRIVE'})
df_ord['HASPAVEDDRIVE'] = df_ord['HASPAVEDDRIVE'].astype('bool')
drop_cols.append('PAVEDDRIVE')
add_cols.append('HASPAVEDDRIVE')


df_ord['LOTSHAPE'] = df_ord['LOTSHAPE'].map(
    {'REG': 1, 'IR1': 0, 'IR2': 0, 'IR3': 0}
)
df_ord = df_ord.rename(columns={'LOTSHAPE': 'HASREGULARLOTSHAPE'})
df_ord['HASREGULARLOTSHAPE'] = df_ord['HASREGULARLOTSHAPE'].astype('bool')
drop_cols.append('LOTSHAPE')
add_cols.append('HASREGULARLOTSHAPE')

# UPDATE ORD_COLS
ord_cols = list(set(ord_cols) - set(drop_cols))
bool_cols = list(set(bool_cols).union(set(add_cols)))

In [2416]:
for col in ord_cols:
    print(df_ord[col].value_counts().sort_index(ascending=False))
    print()

GARAGEFINISH
3    345
2    413
1    580
0     74
Name: count, dtype: int64

BSMTFINTYPE1
3    632
2    354
1    426
Name: count, dtype: int64

KITCHENQUAL
3    672
2    740
Name: count, dtype: int64

FUNCTIONAL
2    1322
1      71
0      19
Name: count, dtype: int64

HEATINGQC
3    959
2    452
0      1
Name: count, dtype: int64

FIREPLACEQU
3    396
2    340
0    676
Name: count, dtype: int64

OVERALLCOND
3      94
2    1240
1      78
Name: count, dtype: int64

ELECTRICAL
3    1300
2     109
1       3
Name: count, dtype: int64

BSMTEXPOSURE
2     133
1    1279
Name: count, dtype: int64

EXTERCOND
3     148
2    1263
0       1
Name: count, dtype: int64

GARAGEQUAL
3      17
2    1318
0      77
Name: count, dtype: int64

BSMTFINTYPE2
3      33
2     133
1    1246
Name: count, dtype: int64

LANDSLOPE
3      13
2      64
1    1335
Name: count, dtype: int64

OVERALLQUAL
3     226
2    1065
1     121
Name: count, dtype: int64

BSMTCOND
3      64
2    1346
0       2
Name: count, dtype: int64

In [2417]:
pd.set_option('display.max_columns', None)
df_ord.head()

,ID,MSSUBCLASS,MSZONING,LOTFRONTAGE,LOTAREA,STREET,ALLEY,HASREGULARLOTSHAPE,LANDCONTOUR,UTILITIES,LOTCONFIG,LANDSLOPE,NEIGHBORHOOD,CONDITION1,CONDITION2,BLDGTYPE,HOUSESTYLE,OVERALLQUAL,OVERALLCOND,YEARBUILT,YEARREMODADD,ROOFSTYLE,ROOFMATL,EXTERIOR1ST,EXTERIOR2ND,MASVNRTYPE,MASVNRAREA,EXTERQUAL,EXTERCOND,FOUNDATION,BSMTQUAL,BSMTCOND,BSMTEXPOSURE,BSMTFINTYPE1,BSMTFINTYPE2,BSMTUNFSF,TOTALBSMTSF,HEATING,HEATINGQC,CENTRALAIR,ELECTRICAL,LOWQUALFINSF,GRLIVAREA,BEDROOMABVGR,KITCHENABVGR,KITCHENQUAL,TOTRMSABVGRD,FUNCTIONAL,FIREPLACES,FIREPLACEQU,GARAGETYPE,GARAGEFINISH,GARAGEQUAL,GARAGECOND,HASPAVEDDRIVE,WOODDECKSF,OPENPORCHSF,ENCLOSEDPORCH,3SSNPORCH,SCREENPORCH,HASFENCE,MISCFEATURE,MISCVAL,SALETYPE,SALECONDITION,SALEPRICE,HASGARAGE,HASPOOL,TOTALBATHS,BSMTFINSF,TOTALFLRSF
0,1,60,RL,65.0,8450.0,PAVE,NA,True,LVL,ALLPUB,INSIDE,1,COLLGCR,NORM,NORM,1FAM,2STORY,2,2,5,5,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,196.0,3,2,PCONC,3,2,1,3,1,150.0,856.0,GASA,3,Y,3,0.0,1710.0,3,1,3,8,2,0,0,ATTCHD,2,2,2,True,0.0,61.0,0.0,0.0,0.0,False,NA,0,WD,NORMAL,208500.0,True,False,3.5,706.0,1710.0
1,2,20,RL,80.0,9600.0,PAVE,NA,True,LVL,ALLPUB,FR2,1,VEENKER,FEEDR,NORM,1FAM,1STORY,2,3,31,31,GABLE,COMPSHG,METALSD,METALSD,None,0.0,2,2,CBLOCK,3,2,2,3,1,284.0,1262.0,GASA,3,Y,3,0.0,1262.0,3,1,2,6,2,1,2,ATTCHD,2,2,2,True,298.0,0.0,0.0,0.0,0.0,False,NA,0,WD,NORMAL,181500.0,True,False,2.5,978.0,1262.0
2,3,60,RL,68.0,11250.0,PAVE,NA,False,LVL,ALLPUB,INSIDE,1,COLLGCR,NORM,NORM,1FAM,2STORY,2,2,7,6,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,162.0,3,2,PCONC,3,2,1,3,1,434.0,920.0,GASA,3,Y,3,0.0,1786.0,3,1,3,6,2,1,2,ATTCHD,2,2,2,True,0.0,42.0,0.0,0.0,0.0,False,NA,0,WD,NORMAL,223500.0,True,False,3.5,486.0,1786.0
3,4,70,RL,60.0,9550.0,PAVE,NA,False,LVL,ALLPUB,CORNER,1,CRAWFOR,NORM,NORM,1FAM,2STORY,2,2,91,36,GABLE,COMPSHG,WD SDNG,WD SHNG,None,0.0,2,2,BRKTIL,2,3,1,3,1,540.0,756.0,GASA,3,Y,3,0.0,1717.0,3,1,3,7,2,1,3,DETCHD,1,2,2,True,0.0,35.0,272.0,0.0,0.0,False,NA,0,WD,ABNORML,140000.0,True,False,2.0,216.0,1717.0
4,5,60,RL,84.0,14260.0,PAVE,NA,False,LVL,ALLPUB,FR2,1,NORIDGE,NORM,NORM,1FAM,2STORY,3,2,8,8,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,350.0,3,2,PCONC,3,2,1,3,1,490.0,1145.0,GASA,3,Y,3,0.0,2198.0,4,1,3,9,2,1,2,ATTCHD,2,2,2,True,192.0,84.0,0.0,0.0,0.0,False,NA,0,WD,NORMAL,250000.0,True,False,3.5,655.0,2198.0


# Quantitative Continuous

In [2418]:
df_cont = df_ord.copy()

## Skewness Analysis and Transformation

In [2419]:
df_skew = df_cont[cont_cols].skew().to_frame('SKEWNESS').sort_values('SKEWNESS', ascending=False)
df_skew

,SKEWNESS
LOTAREA,12.119696
3SSNPORCH,10.379001
LOWQUALFINSF,8.866842
SCREENPORCH,4.042645
ENCLOSEDPORCH,3.104135
MASVNRAREA,2.653450
OPENPORCHSF,2.367689
TOTALBSMTSF,2.163751
WOODDECKSF,1.517519
BSMTFINSF,1.411922


In [2420]:
sym_cols = df_skew[df_skew['SKEWNESS'].abs() < 0.5].index
mod_skew_cols = df_skew[df_skew['SKEWNESS'].abs().between(0.5, 1.0)].index
high_skew_cols = df_skew[df_skew['SKEWNESS'].abs() > 1.0].index

### High skew treatment

#### Zero Presence Analysis (Zero-Heavy High-Skew Variables)

In [2421]:
ZERO_PCT_THRESH = 0.50

df_zero = zero_pct(df_cont, cont_cols)
df_zero = df_zero[df_zero['ZERO_PCT'] > ZERO_PCT_THRESH]
zero_heavy_cols = []

for col in df_zero.index:
    zero_heavy_cols.append(col)


high_skew_cols = list(set(high_skew_cols) - set(zero_heavy_cols))

df_zero = zero_pct(df_cont, zero_heavy_cols)
df_zero

,ZERO_PCT
3SSNPORCH,0.983711
LOWQUALFINSF,0.982295
SCREENPORCH,0.917847
ENCLOSEDPORCH,0.856232
MASVNRAREA,0.586402
WOODDECKSF,0.512040


#### Power Transformation (High Skew Variables)

In [2422]:
pt_map = {}

for col in high_skew_cols:
    pt = PowerTransformer()
    df_cont[col + '_TRANSFORMED'] = pt.fit_transform(df_cont[[col]])[:, 0]
    pt_map[col] = pt


pt_cols = [c for c in df_cont.columns if '_TRANSFORMED' in c]

In [2423]:
# sns.FacetGrid(
#     pd.melt(df_cont, value_vars=pt_cols),
#     col='variable',
#     col_wrap=3,
#     sharex=False,
#     sharey=True,
# ).map(sns.histplot, 'value', stat='density')

#### Zero-Heavy Variables → Boolean Flags

In [2424]:
drop_cols = []
add_cols = []

ZERO_PCT_THRESH = 70

zero_flag_map = {
    col: 'HAS' + col
    for col in df_zero[df_zero['ZERO_PCT'] > ZERO_PCT_THRESH].index
}

for old_col, new_col in zero_flag_map.items():
    df_cont = df_cont.rename(columns={old_col: new_col})
    df_cont[new_col] = np.where(df_cont[new_col] == 0, 0, 1).astype('bool')
    drop_cols.append(old_col)
    add_cols.append(new_col)


zero_heavy_cols = list(set(zero_heavy_cols) - set(drop_cols))
cont_cols = list(set(cont_cols) - set(drop_cols))
bool_cols = list(set(bool_cols).union(set(add_cols)))

#### Remaining Skewed Variables → Log Transform + Presence Flag

In [2425]:
# HANDLE REMAINING VARIABLES (moderate zero presence: bool flag + log1p)
log_flag_map = {col: 'HAS' + col for col in zero_heavy_cols}

for old_col, new_col in log_flag_map.items():
    df_cont[new_col] = np.where(df_cont[old_col] == 0, 0, 1).astype('bool')
    df_cont[old_col + '_LOG'] = np.log1p(df_cont[old_col])
    add_cols.append(new_col)

log_cols = [c for c in df_cont.columns if '_LOG' in c]
bool_cols = list(set(bool_cols).union(set(add_cols)))

log_cols

['WOODDECKSF_LOG',
 '3SSNPORCH_LOG',
 'LOWQUALFINSF_LOG',
 'MASVNRAREA_LOG',
 'ENCLOSEDPORCH_LOG',
 'SCREENPORCH_LOG']

#### Summary: High skew treated Columns

In [2426]:
high_skew_final = list(set(pt_cols).union(set(log_cols)))

### Moderate Skew Treatment

In [2427]:
df_zero = zero_pct(df_cont, mod_skew_cols)
df_zero.round(2)

,ZERO_PCT
BSMTUNFSF,0.06


In [2428]:
bsmt_unf_pt = PowerTransformer()
df_cont['BSMTUNFSF_TRANSFORMED'] = bsmt_unf_pt.fit_transform(df_cont[['BSMTUNFSF']])[:, 0]

#### Summary: Moderate skew treated Columns

In [2429]:
pt_cols = ['BSMTUNFSF_TRANSFORMED']
mod_skew_final = list(set(pt_cols).union(set(log_cols)))

### Approximately Symmetric Treatment

In [2430]:
df_zero = zero_pct(df_cont, sym_cols)
df_zero.round(2)

,ZERO_PCT
LOTFRONTAGE,0.18


In [2431]:
sym_final = []
add_cols = []

df_cont['LOTFRONTAGE_LOG'] = np.log1p(df_cont['LOTFRONTAGE'])
sym_final.append('LOTFRONTAGE_LOG')

df_cont['HASLOTFRONTAGE'] = np.where(df_cont['LOTFRONTAGE'] == 0, 0, 1).astype('bool')
add_cols.append('HASLOTFRONTAGE')
bool_cols = list(set(bool_cols).union(set(add_cols)))

In [2432]:
cont_cols = sym_final

cont_cols = list(set(cont_cols).union(set(mod_skew_final)))
cont_cols = list(set(cont_cols).union(set(high_skew_final)))

In [2433]:
# sns.FacetGrid(
#     pd.melt(df_cont, value_vars=cont_cols),
#     col='variable',
#     col_wrap=4,
#     sharex=False,
#     sharey=True,
# ).map(sns.histplot, 'value', stat='density')

## Outlier treatment

In [2434]:
ZERO_PCT_THRESH = 0.5

df_zero = zero_pct(df_cont, cont_cols)
df_zero = df_zero[df_zero['ZERO_PCT'] > ZERO_PCT_THRESH]

df_zero

,ZERO_PCT
3SSNPORCH_LOG,0.983711
LOWQUALFINSF_LOG,0.982295
SCREENPORCH_LOG,0.917847
ENCLOSEDPORCH_LOG,0.856232
MASVNRAREA_LOG,0.586402
WOODDECKSF_LOG,0.512040


In [2435]:
zero_heavy_final_cols = df_zero.index

In [2436]:
df_out = df_cont.copy()

for col in cont_cols:

    if col in zero_heavy_final_cols:
        non_zero_values = df_cont.loc[df_cont[col] > 0, col]
        Q1 = non_zero_values.quantile(0.25)
        Q3 = non_zero_values.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        mask_outliers = (df_out[col] == 0) | ((df_out[col] > lower_bound) & (df_out[col] < upper_bound))
        df_out = df_out[mask_outliers]

    else:
        Q1 = df_out[col].quantile(0.15)
        Q3 = df_out[col].quantile(0.85)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        mask_outliers = (df_out[col] < lower_bound) | (df_out[col] > upper_bound)
        df_out = df_out[~mask_outliers]


pct_removed = round((df_cont.shape[0] - df_out.shape[0]) / df_cont.shape[0] * 100, 2)
print(f'{pct_removed}% removed')

df_cont = df_out.copy()


6.73% removed


In [2437]:
# sns.FacetGrid(
#     pd.melt(df_cont, value_vars=cont_cols),
#     col='variable',
#     col_wrap=4,
#     sharex=False,
#     sharey=True,
# ).map(sns.histplot, 'value', stat='density')

# Quantitative discrete

In [2438]:
df_disc = df_cont.copy()

In [2439]:
# sns.FacetGrid(
#     pd.melt(df_disc, value_vars=disc_cols),
#     col='variable',
#     col_wrap=4,
#     sharex=False,
#     sharey=True,
# ).map(sns.histplot, 'value')

In [2440]:
df_disc_zero = zero_pct(df_disc, disc_cols.columns)
df_disc_zero = df_disc_zero[df_disc_zero['ZERO_PCT'] > 0]

df_disc_zero

,ZERO_PCT
MISCVAL,0.969628
FIREPLACES,0.464692
YEARREMODADD,0.088838
YEARBUILT,0.045558
BEDROOMABVGR,0.003797
KITCHENABVGR,0.000759


In [2441]:
drop_cols = []
add_cols = []
zero_flag_map = {}


ZERO_PCT_THRESH = 0.50

zero_flag_map = {
    col: 'HAS' + col
    for col in df_disc_zero[df_disc_zero['ZERO_PCT'] > ZERO_PCT_THRESH].index
}

for old_col, new_col in zero_flag_map.items():
    print(old_col, new_col)
    df_disc = df_disc.rename(columns={old_col: new_col})
    df_disc[new_col] = np.where(df_disc[new_col] == 0, 0, 1).astype('bool')
    drop_cols.append(old_col)
    add_cols.append(new_col)


disc_cols = list(set(disc_cols) - set(drop_cols))
bool_cols = list(set(bool_cols).union(set(add_cols)))


MISCVAL HASMISCVAL


## Target

In [2442]:
df_target = df_disc.copy()
df_target['SALEPRICE_LOG'] = np.log1p(df_target[['SALEPRICE']])
target = df_target['SALEPRICE_LOG'] 

In [2443]:
df_target.head()

,ID,MSSUBCLASS,MSZONING,LOTFRONTAGE,LOTAREA,STREET,ALLEY,HASREGULARLOTSHAPE,LANDCONTOUR,UTILITIES,LOTCONFIG,LANDSLOPE,NEIGHBORHOOD,CONDITION1,CONDITION2,BLDGTYPE,HOUSESTYLE,OVERALLQUAL,OVERALLCOND,YEARBUILT,YEARREMODADD,ROOFSTYLE,ROOFMATL,EXTERIOR1ST,EXTERIOR2ND,MASVNRTYPE,MASVNRAREA,EXTERQUAL,EXTERCOND,FOUNDATION,BSMTQUAL,BSMTCOND,BSMTEXPOSURE,BSMTFINTYPE1,BSMTFINTYPE2,BSMTUNFSF,TOTALBSMTSF,HEATING,HEATINGQC,CENTRALAIR,ELECTRICAL,LOWQUALFINSF,GRLIVAREA,BEDROOMABVGR,KITCHENABVGR,KITCHENQUAL,TOTRMSABVGRD,FUNCTIONAL,FIREPLACES,FIREPLACEQU,GARAGETYPE,GARAGEFINISH,GARAGEQUAL,GARAGECOND,HASPAVEDDRIVE,WOODDECKSF,OPENPORCHSF,ENCLOSEDPORCH,3SSNPORCH,SCREENPORCH,HASFENCE,MISCFEATURE,HASMISCVAL,SALETYPE,SALECONDITION,SALEPRICE,HASGARAGE,HASPOOL,TOTALBATHS,BSMTFINSF,TOTALFLRSF,OPENPORCHSF_TRANSFORMED,BSMTFINSF_TRANSFORMED,LOTAREA_TRANSFORMED,TOTALFLRSF_TRANSFORMED,GRLIVAREA_TRANSFORMED,TOTALBSMTSF_TRANSFORMED,HASWOODDECKSF,WOODDECKSF_LOG,HAS3SSNPORCH,3SSNPORCH_LOG,HASLOWQUALFINSF,LOWQUALFINSF_LOG,HASMASVNRAREA,MASVNRAREA_LOG,HASENCLOSEDPORCH,ENCLOSEDPORCH_LOG,HASSCREENPORCH,SCREENPORCH_LOG,BSMTUNFSF_TRANSFORMED,LOTFRONTAGE_LOG,HASLOTFRONTAGE,SALEPRICE_LOG
0,1,60,RL,65.0,8450.0,PAVE,NA,True,LVL,ALLPUB,INSIDE,1,COLLGCR,NORM,NORM,1FAM,2STORY,2,2,5,5,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,196.0,3,2,PCONC,3,2,1,3,1,150.0,856.0,GASA,3,Y,3,0.0,1710.0,3,1,3,8,2,0,0,ATTCHD,2,2,2,True,0.0,61.0,0.0,0.0,0.0,False,NA,False,WD,NORMAL,208500.0,True,False,3.5,706.0,1710.0,0.824046,0.693146,-0.139379,0.534576,0.526899,-0.502024,False,0.000000,False,0.0,False,0.0,True,5.283204,False,0.000000,False,0.0,-0.986479,4.189655,True,12.247699
1,2,20,RL,80.0,9600.0,PAVE,NA,True,LVL,ALLPUB,FR2,1,VEENKER,FEEDR,NORM,1FAM,1STORY,2,3,31,31,GABLE,COMPSHG,METALSD,METALSD,None,0.0,2,2,CBLOCK,3,2,2,3,1,284.0,1262.0,GASA,3,Y,3,0.0,1262.0,3,1,2,6,2,1,2,ATTCHD,2,2,2,True,298.0,0.0,0.0,0.0,0.0,False,NA,False,WD,NORMAL,181500.0,True,False,2.5,978.0,1262.0,-1.086149,0.926719,0.105266,-0.379938,-0.388267,0.587676,True,5.700444,False,0.0,False,0.0,False,0.000000,False,0.000000,False,0.0,-0.514241,4.394449,True,12.109016
2,3,60,RL,68.0,11250.0,PAVE,NA,False,LVL,ALLPUB,INSIDE,1,COLLGCR,NORM,NORM,1FAM,2STORY,2,2,7,6,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,162.0,3,2,PCONC,3,2,1,3,1,434.0,920.0,GASA,3,Y,3,0.0,1786.0,3,1,3,6,2,1,2,ATTCHD,2,2,2,True,0.0,42.0,0.0,0.0,0.0,False,NA,False,WD,NORMAL,223500.0,True,False,3.5,486.0,1786.0,0.646330,0.449810,0.410804,0.665267,0.657048,-0.304801,False,0.000000,False,0.0,False,0.0,True,5.093750,False,0.000000,False,0.0,-0.109676,4.234107,True,12.317171
3,4,70,RL,60.0,9550.0,PAVE,NA,False,LVL,ALLPUB,CORNER,1,CRAWFOR,NORM,NORM,1FAM,2STORY,2,2,91,36,GABLE,COMPSHG,WD SDNG,WD SHNG,None,0.0,2,2,BRKTIL,2,3,1,3,1,540.0,756.0,GASA,3,Y,3,0.0,1717.0,3,1,3,7,2,1,3,DETCHD,1,2,2,True,0.0,35.0,272.0,0.0,0.0,False,NA,False,WD,ABNORML,140000.0,True,False,2.0,216.0,1717.0,0.560644,-0.000456,0.095234,0.546856,0.539135,-0.836425,False,0.000000,False,0.0,False,0.0,False,0.000000,True,5.609472,False,0.0,0.133421,4.110874,True,11.849405
4,5,60,RL,84.0,14260.0,PAVE,NA,False,LVL,ALLPUB,FR2,1,NORIDGE,NORM,NORM,1FAM,2STORY,3,2,8,8,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,350.0,3,2,PCONC,3,2,1,3,1,490.0,1145.0,GASA,3,Y,3,0.0,2198.0,4,1,3,9,2,1,2,ATTCHD,2,2,2,True,192.0,84.0,0.0,0.0,0.0,False,NA,False,WD,NORMAL,250000.0,True,False,3.5,655.0,2198.0,0.978631,0.642277,0.870525,1.288358,1.275383,0.308044,True,5.262690,False,0.0,False,0.0,True,5.860786,False,0.000000,False,0.0,0.022165,4.442651,True,12.429220


# Qualitative nominal

In [2444]:
df_nom = df_target.copy()

encoder = TargetEncoder(cols=nom_cols, smoothing=5.0)
df_nom[nom_cols] = encoder.fit_transform(df_nom[nom_cols], df_nom['SALEPRICE_LOG'])

# Prepare model dataframe

In [2445]:
df_model = df_nom.copy()

In [2446]:
original_to_drop = [
    "LOTAREA",
    "TOTALBSMTSF",
    "GRLIVAREA",
    "LOWQUALFINSF",
    "OPENPORCHSF",
    "MASVNRAREA",
    "WOODDECKSF",
    "ENCLOSEDPORCH",
    "3SSNPORCH",
    "SCREENPORCH",
    "BSMTUNFSF",
    "BSMTFINSF",
    "TOTALFLRSF",
    "LOTFRONTAGE",
    "SALEPRICE",
    "UTILITIES",
]

df_model = df_model.drop(columns=original_to_drop)

In [2447]:
def remove_high_vif(df, threshold, exclude_cols=None):
    X = df.copy()
    

    if exclude_cols:
        X = X.drop(columns=[c for c in exclude_cols if c in X.columns])

    bool_cols = X.select_dtypes(include=['bool']).columns
    X = X.drop(columns=bool_cols)
    
    X = add_constant(X)
    X = X.astype(float)
    
    dropped_cols = []
    
    while True:
        vif_data = pd.DataFrame()
        vif_data['VARIAVEL'] = X.columns
        vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        vif_data = vif_data[vif_data['VARIAVEL'] != 'const']
        
        max_vif = vif_data['VIF'].max()
        
        if max_vif <= threshold:
            break
        
        var_to_drop = vif_data.sort_values('VIF', ascending=False).iloc[0]['VARIAVEL']
        print(f"Removendo '{var_to_drop}' (VIF={max_vif:.2f})")
        X = X.drop(columns=[var_to_drop])
        dropped_cols.append(var_to_drop)
    
    return dropped_cols

cols_to_drop = remove_high_vif(df_model, threshold=4, exclude_cols=['ID', 'SALEPRICE_LOG'])
print(cols_to_drop)

df_model = df_model.drop(columns=cols_to_drop)

Removendo 'TOTALFLRSF_TRANSFORMED' (VIF=496.81)
Removendo 'GARAGECOND' (VIF=13.34)
Removendo 'BSMTFINSF_TRANSFORMED' (VIF=11.37)
Removendo 'EXTERIOR2ND' (VIF=10.90)
Removendo 'SALETYPE' (VIF=8.34)
Removendo 'YEARBUILT' (VIF=8.04)
Removendo 'GRLIVAREA_TRANSFORMED' (VIF=5.97)
Removendo 'MASVNRTYPE' (VIF=5.70)
Removendo 'FIREPLACEQU' (VIF=4.49)
['TOTALFLRSF_TRANSFORMED', 'GARAGECOND', 'BSMTFINSF_TRANSFORMED', 'EXTERIOR2ND', 'SALETYPE', 'YEARBUILT', 'GRLIVAREA_TRANSFORMED', 'MASVNRTYPE', 'FIREPLACEQU']


In [ ]:
numeric_columns = df_model.select_dtypes(include=['int64', 'float64']).columns

df_model_corr= df_model[numeric_columns].corr()
df_model_corr = df_model_corr.drop(index=['ID', 'SALEPRICE_LOG'], columns='ID')

df_model_corr = pd.DataFrame(df_model_corr['SALEPRICE_LOG']).rename(columns={'SALEPRICE_LOG': 'CORRELATION'})
df_model_corr['CORRELATION'] = df_model_corr['CORRELATION'].abs()
df_model_corr = df_model_corr.sort_values(by=['CORRELATION'], ascending=False)

columns_to_drop = (df_model_corr[df_model_corr['CORRELATION'] < 0.3]).index

df_model = df_model.drop(columns=columns_to_drop)

In [2449]:
df_model.info()

<class 'pandas.DataFrame'>
Index: 1317 entries, 0 to 1410
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       1317 non-null   int64  
 1   MSSUBCLASS               1317 non-null   float64
 2   MSZONING                 1317 non-null   float64
 3   HASREGULARLOTSHAPE       1317 non-null   bool   
 4   NEIGHBORHOOD             1317 non-null   float64
 5   HOUSESTYLE               1317 non-null   float64
 6   OVERALLQUAL              1317 non-null   int64  
 7   YEARREMODADD             1317 non-null   int64  
 8   EXTERIOR1ST              1317 non-null   float64
 9   EXTERQUAL                1317 non-null   int64  
 10  FOUNDATION               1317 non-null   float64
 11  BSMTQUAL                 1317 non-null   int64  
 12  HEATINGQC                1317 non-null   int64  
 13  CENTRALAIR               1317 non-null   float64
 14  KITCHENQUAL              1317 non-null  

In [2450]:
df_model.head()

,ID,MSSUBCLASS,MSZONING,HASREGULARLOTSHAPE,NEIGHBORHOOD,HOUSESTYLE,OVERALLQUAL,YEARREMODADD,EXTERIOR1ST,EXTERQUAL,FOUNDATION,BSMTQUAL,HEATINGQC,CENTRALAIR,KITCHENQUAL,TOTRMSABVGRD,FIREPLACES,GARAGETYPE,GARAGEFINISH,GARAGEQUAL,HASPAVEDDRIVE,HASFENCE,HASMISCVAL,SALECONDITION,HASGARAGE,HASPOOL,TOTALBATHS,OPENPORCHSF_TRANSFORMED,LOTAREA_TRANSFORMED,TOTALBSMTSF_TRANSFORMED,HASWOODDECKSF,WOODDECKSF_LOG,HAS3SSNPORCH,HASLOWQUALFINSF,HASMASVNRAREA,MASVNRAREA_LOG,HASENCLOSEDPORCH,HASSCREENPORCH,HASLOTFRONTAGE,SALEPRICE_LOG
0,1,12.340706,12.099821,True,12.164080,12.215481,2,5,12.219079,3,12.272285,3,3,12.077915,3,8,0,12.169156,2,2,True,False,False,12.022480,True,False,3.5,0.824046,-0.139379,-0.502024,False,0.000000,False,False,True,5.283204,False,False,True,12.247699
1,2,12.067314,12.099821,True,12.074823,12.010494,2,31,11.878527,2,11.883880,3,3,12.077915,2,6,1,12.169156,2,2,True,False,False,12.022480,True,False,2.5,-1.086149,0.105266,0.587676,True,5.700444,False,False,False,0.000000,False,False,True,12.109016
2,3,12.340706,12.099821,False,12.164080,12.215481,2,6,12.219079,3,12.272285,3,3,12.077915,3,6,1,12.169156,2,2,True,False,False,12.022480,True,False,3.5,0.646330,0.410804,-0.304801,False,0.000000,False,False,True,5.093750,False,False,True,12.317171
3,4,11.972299,12.099821,False,12.224809,12.215481,2,36,11.844305,2,11.712360,2,3,12.077915,3,7,1,11.785369,1,2,True,False,False,11.831203,True,False,2.0,0.560644,0.095234,-0.836425,False,0.000000,False,False,False,0.000000,True,False,True,11.849405
4,5,12.340706,12.099821,False,12.659058,12.215481,3,8,12.219079,3,12.272285,3,3,12.077915,3,9,1,12.169156,2,2,True,False,False,12.022480,True,False,3.5,0.978631,0.870525,0.308044,True,5.262690,False,False,True,5.860786,False,False,True,12.429220
